# Evaluación de SAE con Métricas NeurIPS 2024

Implementación de las métricas Coverage y Reconstruction del paper:
"Measuring Progress in Dictionary Learning for Language Model Interpretability with Board Game Models"
(Karvonen et al., NeurIPS 2024)

## Datos disponibles:
- 200 partidas × 59 movimientos = 11,800 posiciones
- Activaciones SAE de capa 6
- Ground truth de 198 BSPs por posición

## Métricas:
1. **Coverage**: ¿Qué % de BSPs puede detectar el SAE con alta precisión?


In [4]:
pip install scikit-learn

   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/8.1 MB 6.3 MB/s eta 0:00:02
   -------------- ------------------------- 2.9/8.1 MB 7.6 MB/s eta 0:00:01
   ------------------------ --------------- 5.0/8.1 MB 8.9 MB/s eta 0:00:01
   ------------------------------------- -- 7.6/8.1 MB 9.8 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 9.6 MB/s  0:00:00
   ---------------------------------------- 0.0/36.4 MB ? eta -:--:--
   -- ------------------------------------- 2.1/36.4 MB 10.7 MB/s eta 0:00:04
   ----- ---------------------------------- 4.7/36.4 MB 11.4 MB/s eta 0:00:03
   ------- -------------------------------- 6.8/36.4 MB 11.3 MB/s eta 0:00:03
   ------------ --------------------------- 11.3/36.4 MB 14.1 MB/s eta 0:00:02
   --------------------- ------------------ 19.1/36.4 MB 18.9 MB/s eta 0:00:01
   ----------------------------- ---------- 26.5/36.4 MB 22.1 MB/s eta 0:00:01
   ----------

In [1]:
import numpy as np
import torch
import sys
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score

# Proyecto root: dos niveles arriba de notebooks/
project_root = Path('../..').resolve()
sys.path.insert(0, str(project_root))

print(f"Proyecto: {project_root}")

Proyecto: C:\Users\Esposa\Documents\Repos\othello_world


## 1. Cargar Datos

In [2]:
# Cargar activaciones del modelo (pre-SAE)
activations_path = project_root / "sae" / "activations" / "data" / "layer5_200games.npy"
activations = np.load(activations_path)

print(f"Activaciones cargadas:")
print(f"  Shape: {activations.shape}")
print(f"  Tipo: {activations.dtype}")
print(f"  Memoria: {activations.nbytes / (1024**2):.2f} MB")

Activaciones cargadas:
  Shape: (11800, 512)
  Tipo: float32
  Memoria: 23.05 MB


In [3]:
# Cargar ground truth de BSPs
bsp_gt_path = project_root / "sae" / "metrics" / "data" / "bsp_ground_truth_200games.npy"
bsp_names_path = project_root / "sae" / "metrics" / "data" / "bsp_ground_truth_200games.names.npy"

bsp_ground_truth = np.load(bsp_gt_path)
bsp_names = np.load(bsp_names_path, allow_pickle=True)

print(f"\nGround truth BSPs:")
print(f"  Shape: {bsp_ground_truth.shape}")
print(f"  Total BSPs: {len(bsp_names)}")
print(f"  Primeras 5 BSPs: {bsp_names[:5].tolist()}")
print(f"  Últimas 5 BSPs: {bsp_names[-5:].tolist()}")


Ground truth BSPs:
  Shape: (11800, 198)
  Total BSPs: 198
  Primeras 5 BSPs: ['BSPA10', 'BSPA11', 'BSPA12', 'BSPA20', 'BSPA21']
  Últimas 5 BSPs: ['BSP_TERMINADA', 'BSP_GANE', 'BSP_PERDI', 'BSP_EMPATE', 'BSP_EN_CURSO']


## 2. Cargar Modelo SAE

In [4]:
from sae.model.sae import SparseAutoencoder

# Configuración del SAE (debe coincidir con el entrenamiento)
input_dim = 512  # Dimensión de activaciones de capa 5
hidden_dim = 16384  # 32x expansion

# Cargar modelo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sae = SparseAutoencoder(input_dim, hidden_dim).to(device)

model_path = project_root / "sae" / "model" / "saved_model" / "sae_othello_best.pt"
checkpoint = torch.load(model_path, map_location=device)
sae.load_state_dict(checkpoint['model_state_dict'])
sae.eval()

print(f"SAE cargado:")
print(f"  Input dim: {input_dim}")
print(f"  Hidden dim: {hidden_dim}")
print(f"  Device: {device}")
print(f"  Expansion: {hidden_dim/input_dim}x")
print(f"  Epoch entrenado: {checkpoint['epoch']}")
print(f"  Val MSE: {checkpoint['val_mse']:.6f}")

SAE cargado:
  Input dim: 512
  Hidden dim: 16384
  Device: cuda
  Expansion: 32.0x
  Epoch entrenado: 29
  Val MSE: 0.011182


## 3. Extraer Features del SAE

In [5]:
# Convertir activaciones a tensor
activations_tensor = torch.from_numpy(activations).float().to(device)

# Extraer features del SAE (activaciones de la capa oculta)
with torch.no_grad():
    # Forward pass por el encoder
    encoded = torch.relu(sae.encoder(activations_tensor))
    
# Convertir a numpy
sae_features = encoded.cpu().numpy()

print(f"Features SAE extraídas:")
print(f"  Shape: {sae_features.shape}")
print(f"  Sparsity: {np.mean(sae_features == 0):.2%}")
print(f"  Activaciones promedio por posición: {np.mean(np.sum(sae_features > 0, axis=1)):.1f}")

Features SAE extraídas:
  Shape: (11800, 16384)
  Sparsity: 96.54%
  Activaciones promedio por posición: 567.5


## 4. Preparar BSPs para Coverage

Coverage usa **solo las 128 BSPs de piezas** (sin vacías):
- 64 BSPs de "casilla X es mía" (terminan en '1')
- 64 BSPs de "casilla X es del oponente" (terminan en '2')

In [6]:
# Filtrar solo BSPs de piezas (sin vacías, sin especiales)
bsp_pieces_indices = []
bsp_pieces_names = []

for i, name in enumerate(bsp_names):
    # Solo BSPs de casillas que NO son vacías (no terminan en '0')
    # Y no son especiales (no empiezan con 'BSP_')
    if len(name) == 6 and name.startswith('BSP') and not name.endswith('0'):
        bsp_pieces_indices.append(i)
        bsp_pieces_names.append(name)

bsp_pieces_indices = np.array(bsp_pieces_indices)
bsp_pieces_gt = bsp_ground_truth[:, bsp_pieces_indices]

print(f"BSPs para Coverage:")
print(f"  Total BSPs de piezas: {len(bsp_pieces_indices)}")
print(f"  Shape de ground truth: {bsp_pieces_gt.shape}")
print(f"  Ejemplos: {bsp_pieces_names[:10]}")

BSPs para Coverage:
  Total BSPs de piezas: 128
  Shape de ground truth: (11800, 128)
  Ejemplos: [np.str_('BSPA11'), np.str_('BSPA12'), np.str_('BSPA21'), np.str_('BSPA22'), np.str_('BSPA31'), np.str_('BSPA32'), np.str_('BSPA41'), np.str_('BSPA42'), np.str_('BSPA51'), np.str_('BSPA52')]


## 5. Implementar Métrica Coverage

**Algoritmo:**
```
Para cada BSP:
  Para cada feature del SAE:
    Para cada threshold t ∈ {0, 0.1, 0.2, ..., 0.9}:
      - Binarizar feature: predictions = (feature > t * feature_max)
      - Calcular F1(predictions, bsp_ground_truth)
      - Guardar si es el mejor F1
  
Coverage = promedio de los mejores F1 de todas las BSPs
```

In [7]:
def fast_f1_score(y_true, y_pred):
    """
    F1-score vectorizado sin sklearn.
    ~10x más rápido para arrays binarios.
    """
    y_true = y_true.astype(bool)
    y_pred = y_pred.astype(bool)
    
    tp = np.sum(y_true & y_pred)
    fp = np.sum(~y_true & y_pred)
    fn = np.sum(y_true & ~y_pred)
    
    if tp == 0:
        return 0.0
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    
    if precision + recall == 0:
        return 0.0
    
    return 2 * (precision * recall) / (precision + recall)


def calculate_coverage(sae_features, bsp_ground_truth, thresholds=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]):
    """
    Calcula la métrica Coverage (OPTIMIZADO).
    
    Args:
        sae_features: Array (n_positions, n_features) con activaciones del SAE
        bsp_ground_truth: Array (n_positions, n_bsps) con ground truth
        thresholds: Lista de thresholds a probar (como fracción del máximo)
    
    Returns:
        coverage: Score promedio
        best_f1s: Lista de mejores F1 por BSP
        best_features: Lista de mejores features por BSP
        best_thresholds: Lista de mejores thresholds por BSP
    """
    n_positions, n_features = sae_features.shape
    n_bsps = bsp_ground_truth.shape[1]
    
    best_f1s = []
    best_features = []
    best_thresholds = []
    
    print(f"Calculando Coverage (OPTIMIZADO) sobre {n_bsps} BSPs con {n_features} features...")
    print(f"Thresholds a probar: {thresholds}")
    print(f"Total de evaluaciones: {n_bsps * n_features * len(thresholds):,}")
    print()
    
    # PRE-CALCULAR MÁXIMOS DE FEATURES (optimización clave)
    print("Pre-calculando máximos de features...")
    ########################################################
    f_max_per_feature = np.max(sae_features, axis=0)  # (n_features,)
    active_features = np.where(f_max_per_feature > 0)[0]
    print(f"✓ Máximos calculados. Features activas: {len(active_features)}/{n_features}")
    print()
    
    # Para cada BSP
    for bsp_idx in tqdm(range(n_bsps), desc="Procesando BSPs"):
        bsp_labels = bsp_ground_truth[:, bsp_idx].astype(bool)
        
        # Skip si la BSP nunca está activa
        if not bsp_labels.any():
            best_f1s.append(0.0)
            best_features.append(-1)
            best_thresholds.append(-1)
            continue
        
        # Buscar la mejor (feature, threshold)
        best_f1 = 0.0
        best_feature_idx = -1
        best_threshold = -1
        
        # Solo procesar features activas
        for feature_idx in active_features:
            feature_activations = sae_features[:, feature_idx]
            f_max = f_max_per_feature[feature_idx]
            

            # Para cada threshold
            for t in thresholds:
                # Binarizar feature
                ########################################################
                predictions = (feature_activations > t * f_max) 
                
                # Calcular F1 rápido
                f1 = fast_f1_score(bsp_labels, predictions)
                
                # Actualizar si es mejor
                if f1 > best_f1:
                    best_f1 = f1
                    best_feature_idx = feature_idx
                    best_threshold = t
                
                # Early stopping: si ya es perfecto, no buscar más
                if f1 >= 0.999:
                    break
            
            # Early stopping: si ya es perfecto, no revisar más features
            if best_f1 >= 0.999:
                break
        
        best_f1s.append(best_f1)
        best_features.append(best_feature_idx)
        best_thresholds.append(best_threshold)
    
    # Coverage = macro-average de los mejores F1
    coverage = np.mean(best_f1s)
    
    return coverage, best_f1s, best_features, best_thresholds

In [8]:
# Calcular Coverage
coverage, best_f1s, best_features, best_thresholds = calculate_coverage(
    sae_features, 
    bsp_pieces_gt
)

print("\n" + "="*60)
print("RESULTADO COVERAGE")
print("="*60)
print(f"Coverage Score: {coverage:.4f}")
print(f"Objetivo (paper): 0.52")
print(f"Diferencia: {coverage - 0.52:.4f}")
print("="*60)

Calculando Coverage (OPTIMIZADO) sobre 128 BSPs con 16384 features...
Thresholds a probar: [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
Total de evaluaciones: 20,971,520

Pre-calculando máximos de features...
✓ Máximos calculados. Features activas: 9626/16384



Procesando BSPs: 100%|██████████| 128/128 [43:42<00:00, 20.49s/it]


RESULTADO COVERAGE
Coverage Score: 0.4736
Objetivo (paper): 0.52
Diferencia: -0.0464


## 6. Análisis de Resultados

In [9]:
# Estadísticas de F1 scores
best_f1s_array = np.array(best_f1s)

print("Distribución de F1 scores:")
print(f"  Mínimo: {np.min(best_f1s_array):.4f}")
print(f"  Máximo: {np.max(best_f1s_array):.4f}")
print(f"  Media: {np.mean(best_f1s_array):.4f}")
print(f"  Mediana: {np.median(best_f1s_array):.4f}")
print(f"  Std: {np.std(best_f1s_array):.4f}")
print()
print(f"BSPs con F1 > 0.8: {np.sum(best_f1s_array > 0.8)} ({np.mean(best_f1s_array > 0.8):.1%})")
print(f"BSPs con F1 > 0.5: {np.sum(best_f1s_array > 0.5)} ({np.mean(best_f1s_array > 0.5):.1%})")
print(f"BSPs con F1 < 0.2: {np.sum(best_f1s_array < 0.2)} ({np.mean(best_f1s_array < 0.2):.1%})")

Distribución de F1 scores:
  Mínimo: 0.3609
  Máximo: 0.7862
  Media: 0.4736
  Mediana: 0.4519
  Std: 0.0988

BSPs con F1 > 0.8: 0 (0.0%)
BSPs con F1 > 0.5: 43 (33.6%)
BSPs con F1 < 0.2: 0 (0.0%)


In [10]:
# Top 10 BSPs mejor detectadas
top_indices = np.argsort(best_f1s_array)[-10:][::-1]

print("\nTop 10 BSPs mejor detectadas:")
print("="*60)
for idx in top_indices:
    print(f"{bsp_pieces_names[idx]:8s} | F1: {best_f1s[idx]:.4f} | Feature: {best_features[idx]:5d} | Threshold: {best_thresholds[idx]:.1f}")


Top 10 BSPs mejor detectadas:
BSPA12   | F1: 0.7862 | Feature:  9721 | Threshold: 0.3
BSPH82   | F1: 0.7855 | Feature: 10178 | Threshold: 0.3
BSPA82   | F1: 0.7512 | Feature:  4712 | Threshold: 0.3
BSPE41   | F1: 0.6910 | Feature:   924 | Threshold: 0.0
BSPD41   | F1: 0.6905 | Feature:  4729 | Threshold: 0.0
BSPE51   | F1: 0.6899 | Feature:  4519 | Threshold: 0.0
BSPD51   | F1: 0.6883 | Feature: 12979 | Threshold: 0.0
BSPD31   | F1: 0.6276 | Feature: 14425 | Threshold: 0.0
BSPC51   | F1: 0.6253 | Feature:  6034 | Threshold: 0.0
BSPE61   | F1: 0.6187 | Feature: 12831 | Threshold: 0.0


In [11]:
# Bottom 10 BSPs peor detectadas
bottom_indices = np.argsort(best_f1s_array)[:10]

print("\nBottom 10 BSPs peor detectadas:")
print("="*60)
for idx in bottom_indices:
    print(f"{bsp_pieces_names[idx]:8s} | F1: {best_f1s[idx]:.4f} | Feature: {best_features[idx]:5d} | Threshold: {best_thresholds[idx]:.1f}")


Bottom 10 BSPs peor detectadas:
BSPH72   | F1: 0.3609 | Feature:  6955 | Threshold: 0.2
BSPC82   | F1: 0.3628 | Feature:  5894 | Threshold: 0.2
BSPA52   | F1: 0.3633 | Feature:  6955 | Threshold: 0.1
BSPH62   | F1: 0.3638 | Feature:  6955 | Threshold: 0.2
BSPE82   | F1: 0.3645 | Feature:   642 | Threshold: 0.1
BSPH52   | F1: 0.3647 | Feature:  6955 | Threshold: 0.2
BSPE12   | F1: 0.3654 | Feature:  6955 | Threshold: 0.2
BSPB82   | F1: 0.3662 | Feature:  6955 | Threshold: 0.2
BSPH32   | F1: 0.3689 | Feature:  6955 | Threshold: 0.2
BSPH42   | F1: 0.3695 | Feature:  1515 | Threshold: 0.3


In [12]:
# Análisis de thresholds más usados
best_thresholds_array = np.array(best_thresholds)
unique_thresholds, counts = np.unique(best_thresholds_array, return_counts=True)

print("\nDistribución de thresholds óptimos:")
print("="*60)
for t, count in zip(unique_thresholds, counts):
    print(f"Threshold {t:.1f}: {count:3d} BSPs ({count/len(best_thresholds)*100:.1f}%)")


Distribución de thresholds óptimos:
Threshold 0.0:  57 BSPs (44.5%)
Threshold 0.1:  19 BSPs (14.8%)
Threshold 0.2:  41 BSPs (32.0%)
Threshold 0.3:  10 BSPs (7.8%)
Threshold 0.4:   1 BSPs (0.8%)


## Resumen

**Coverage implementado ✓**

- Evaluamos 128 BSPs de piezas (sin vacías)
- Probamos 10 thresholds por feature
- Usamos macro-average sobre BSPs
- Siguiente: Implementar Reconstruction metric